# CyberSentry — 01 · Data Cleaning (CIC-IDS2017)

**Project:** Intelligent Cybersecurity Threat Analysis Using ML, DL and Generative AI
**Stage:** Data loading + cleaning (before feature selection and modelling)

**Input:** the 8 original CSVs of CIC-IDS2017 (*MachineLearningCSV* version, UNB Canadian Institute for Cybersecurity) in `data/raw/`
**Output:** one cleaned file `data/processed/cicids2017_clean.parquet`

What this notebook does:
1. Loads the 8 CSVs one at a time (memory-efficient) and merges them
2. Fixes known dataset problems: spaces in column names, garbled Web Attack labels, a duplicated column, empty rows, infinite values, missing values, duplicate rows
3. Reports constant columns (listed only, not dropped — team decision)
4. Adds `binary_label` (0 = BENIGN, 1 = attack) for the ML anomaly-detection stage, and keeps the multi-class `Label` for the DL stage
5. Logs rows removed at every step and label counts before/after, for the project report

> Raw files in `data/raw/` are never modified.

## 1. Setup

In [ ]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

# Works whether the notebook is run from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = PROC_DIR / "cicids2017_clean.parquet"

csv_files = sorted(RAW_DIR.glob("*.csv"))
print(f"Project root : {ROOT.resolve()}")
print(f"Found {len(csv_files)} CSV files in {RAW_DIR}")
for f in csv_files:
    print(f"  {f.name:<70} {f.stat().st_size / 1e6:8.1f} MB")
assert len(csv_files) == 8, "Expected 8 CIC-IDS2017 CSV files in data/raw/"

## 2. Helper functions

- **`fix_label`** — In the original files the Web Attack labels contain a broken dash character (byte `0x96`), so they show up as `Web Attack � Brute Force`. We normalise them to `Web Attack - Brute Force`, `Web Attack - XSS`, `Web Attack - Sql Injection`.
- **`downcast`** — The CSVs load as 64-bit numbers. Converting to 32-bit roughly halves memory use, which matters with ~2.8 million rows. Float values are kept as `float32` (enough precision for ML); integers are only downcast when every value fits.
- **`short_name`** — a short "day" tag for each file, stored in a `source_file` column so we always know where a row came from.

In [ ]:
WEB_ATTACK_RE = re.compile(r"^Web Attack\s*[^A-Za-z0-9]*\s*", flags=re.IGNORECASE)

def fix_label(label: str) -> str:
    label = str(label).strip()
    if label.lower().startswith("web attack"):
        label = WEB_ATTACK_RE.sub("Web Attack - ", label)
    return label

def downcast(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        dt = df[col].dtype
        if pd.api.types.is_float_dtype(dt):
            df[col] = df[col].astype(np.float32)
        elif pd.api.types.is_integer_dtype(dt):
            df[col] = pd.to_numeric(df[col], downcast="integer")
    return df

def short_name(path: Path) -> str:
    return path.name.replace(".pcap_ISCX.csv", "").replace("-WorkingHours", "").replace("-workingHours", "")

## 3. Load and merge the 8 CSVs

Each file is read, cleaned at the column level and downcast **before** merging, so we never hold all the raw 64-bit data in memory at once.

The files are read with `latin-1` encoding because the Web Attack file contains a non-UTF-8 byte that makes the default reader crash.

In [ ]:
frames = []
load_log = []
t0 = time.time()

for path in csv_files:
    df = pd.read_csv(path, encoding="latin-1", low_memory=False)
    df.columns = df.columns.str.strip()                     # " Label" -> "Label"
    rows_read = len(df)
    df = df.dropna(how="all")                               # fully empty rows (Thursday file has many)
    df = downcast(df)
    df["source_file"] = short_name(path)
    frames.append(df)
    load_log.append({"file": short_name(path), "rows_read": rows_read,
                     "empty_rows_dropped": rows_read - len(df), "columns": df.shape[1] - 1})
    print(f"Loaded {short_name(path):<35} {rows_read:>9,} rows")

data = pd.concat(frames, ignore_index=True)
del frames

print(f"\nMerged shape: {data.shape}  |  memory: {data.memory_usage(deep=True).sum() / 1e9:.2f} GB  |  {time.time() - t0:.0f}s")
pd.DataFrame(load_log)

## 4. First look at the merged data

In [ ]:
print("Columns:", data.shape[1])
print(list(data.columns))
data.head()

## 5. Fix label names

Label values are stripped of extra spaces and the garbled Web Attack names are repaired. Below: label counts **before cleaning** (raw, after only fixing names).

In [ ]:
raw_unique = sorted(data["Label"].astype(str).unique())
data["Label"] = data["Label"].map(fix_label)
print("Labels before fix:", raw_unique)
print("Labels after fix :", sorted(data["Label"].unique()))

labels_before = data["Label"].value_counts()
labels_before.to_frame("count_before")

## 6. Cleaning steps

Every step records how many rows it removed, so the numbers can go straight into the report.

In [ ]:
cleaning_log = []
def log_step(step, rows_before, rows_after, note=""):
    cleaning_log.append({"step": step, "rows_before": rows_before, "rows_after": rows_after,
                         "rows_removed": rows_before - rows_after, "note": note})
    print(f"{step:<45} removed {rows_before - rows_after:>9,} rows  ->  {rows_after:,} left")

feature_cols = [c for c in data.columns if c not in ("Label", "source_file")]

### 6.1 Duplicated column `Fwd Header Length.1`
CIC-IDS2017 contains `Fwd Header Length` twice (pandas renames the second one to `Fwd Header Length.1`). We confirm it is identical and drop it.

In [ ]:
if "Fwd Header Length.1" in data.columns:
    same = data["Fwd Header Length"].equals(data["Fwd Header Length.1"])
    print("Fwd Header Length.1 identical to Fwd Header Length:", same)
    data = data.drop(columns=["Fwd Header Length.1"])
    print("Dropped 'Fwd Header Length.1'")
else:
    print("No duplicated 'Fwd Header Length.1' column found")

feature_cols = [c for c in data.columns if c not in ("Label", "source_file")]
print("Feature columns now:", len(feature_cols))

### 6.2 Infinite values → NaN
Some flow-rate features (e.g. `Flow Bytes/s`, `Flow Packets/s`) are infinite when a flow lasts 0 microseconds (division by zero during feature extraction).

In [ ]:
num = data[feature_cols].select_dtypes(include=[np.number])
inf_counts = np.isinf(num).sum()
inf_counts = inf_counts[inf_counts > 0]
print("Columns with infinite values:")
print(inf_counts.to_string() if len(inf_counts) else "  none")

data[num.columns] = num.replace([np.inf, -np.inf], np.nan)
del num

### 6.3 Missing values (NaN)
After the step above, rows with any missing feature value are dropped. They are a tiny fraction of the dataset.

In [ ]:
nan_counts = data[feature_cols].isna().sum()
nan_counts = nan_counts[nan_counts > 0]
print("Columns with missing values:")
print(nan_counts.to_string() if len(nan_counts) else "  none")

before = len(data)
data = data.dropna(subset=feature_cols + ["Label"])
log_step("Drop rows with NaN / infinite values", before, len(data))

### 6.4 Exact duplicate rows
A flow with exactly the same feature values **and** the same label is counted once. The `source_file` column is ignored here so duplicates across days are also caught.

In [ ]:
before = len(data)
data = data.drop_duplicates(subset=feature_cols + ["Label"])
log_step("Drop exact duplicate rows", before, len(data))
data = data.reset_index(drop=True)

### 6.5 Constant columns (report only)
Columns with a single unique value carry no information for a model. They are **listed, not dropped** — the team decides during feature selection.

In [ ]:
constant_cols = [c for c in feature_cols if c in data.columns and data[c].nunique(dropna=False) <= 1]
print(f"{len(constant_cols)} constant columns:")
for c in constant_cols:
    print("  -", c, "=", data[c].iloc[0])

### 6.6 Negative values (report only)
Some features (e.g. `Init_Win_bytes_forward`) use `-1` to mean "not available". This is a known quirk of CIC-IDS2017. We only report it here; handling is decided in preprocessing.

In [ ]:
neg = (data[feature_cols].select_dtypes(include=[np.number]) < 0).sum()
neg = neg[neg > 0].sort_values(ascending=False)
print("Columns containing negative values (count of rows):")
print(neg.to_string() if len(neg) else "  none")

## 7. Labels for the pipeline
- `Label` — original multi-class label (used by the **DL attack-type classifier**)
- `binary_label` — `0` = BENIGN, `1` = any attack (used by the **ML anomaly detector**)

In [ ]:
data["binary_label"] = (data["Label"] != "BENIGN").astype(np.int8)
data["Label"] = data["Label"].astype("category")
data["source_file"] = data["source_file"].astype("category")

print(data["binary_label"].value_counts().rename({0: "BENIGN (0)", 1: "ATTACK (1)"}).to_string())

## 8. Before vs after — label distribution

In [ ]:
labels_after = data["Label"].value_counts()
summary = pd.DataFrame({"before": labels_before, "after": labels_after}).fillna(0).astype(int)
summary["removed"] = summary["before"] - summary["after"]
summary["% of clean data"] = (summary["after"] / summary["after"].sum() * 100).round(4)
summary = summary.sort_values("after", ascending=False)
summary.loc["TOTAL"] = [summary["before"].sum(), summary["after"].sum(), summary["removed"].sum(), 100.0]
summary = summary.astype({"before": int, "after": int, "removed": int})
summary

In [ ]:
log_df = pd.DataFrame(cleaning_log)
total_read = sum(r["rows_read"] for r in load_log)
empty = sum(r["empty_rows_dropped"] for r in load_log)
log_df = pd.concat([pd.DataFrame([{"step": "Drop fully empty rows (at load)", "rows_before": total_read,
                                   "rows_after": total_read - empty, "rows_removed": empty, "note": ""}]),
                    log_df], ignore_index=True)
log_df

## 9. Final checks and save

In [ ]:
num_cols = data.select_dtypes(include=[np.number]).columns
assert data[feature_cols].isna().sum().sum() == 0, "NaN values remain"
assert not np.isinf(data[num_cols].to_numpy(dtype=np.float64)).any(), "Infinite values remain"
assert not data.duplicated(subset=feature_cols + ["Label"]).any(), "Duplicates remain"
print("All checks passed")

data.to_parquet(OUT_FILE, index=False)
print(f"Saved {OUT_FILE}  ({OUT_FILE.stat().st_size / 1e6:.1f} MB)")
print(f"Final shape: {data.shape[0]:,} rows x {data.shape[1]} columns")

In [ ]:
# Sanity check: reload the saved file
check = pd.read_parquet(OUT_FILE)
print(check.shape)
print(check.dtypes.value_counts().to_string())
del check

## 10. Summary

*(Fill in the numbers from the outputs above before showing this to the guide.)*

- **Raw data:** 8 CSV files, ~2.83 million flows, 79 columns (78 features + `Label`).
- **Cleaning:** dropped the duplicated `Fwd Header Length.1` column, removed empty rows, rows with infinite/missing values, and exact duplicate rows. See the cleaning log table for exact counts per step.
- **Final dataset:** see section 9 for final row × column count. Saved as `data/processed/cicids2017_clean.parquet`.
- **Labels:** 15 classes (BENIGN + 14 attack types), plus `binary_label` for anomaly detection.
- **Constant columns:** listed in 6.5, to be dropped during feature selection.

### The imbalance problem (in plain words)
Most of the traffic in CIC-IDS2017 is normal. Roughly 8 out of 10 flows are BENIGN, and some attacks are extremely rare (Heartbleed, Infiltration and SQL Injection have only a few dozen samples each, compared to over two million benign flows).

This means:
- A model that predicts "BENIGN" for everything would still score ~80% **accuracy** while catching zero attacks, so accuracy alone is misleading.
- We must evaluate with **precision, recall, F1, FPR/FNR and confusion matrices**, per class.
- Rare classes may need handling later: class weights, resampling (e.g. SMOTE on the training set only), or merging very rare classes into broader groups. This will be decided with the team during the modelling stage.